In [28]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [30]:
warnings.filterwarnings("ignore")

DATA_PATH = "youtube_ad_revenue_dataset.csv"
OUTPUT_DIR = "outputs"
TARGET_COL = "ad_revenue_usd"


In [31]:
def create_output_dir():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

In [32]:
def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found at: {file_path}")
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    return df

In [33]:

def basic_inspection(df):
    print("\n--- DATASET INFO ---")
    print(df.info())
    print("\n--- FIRST 5 ROWS ---")
    print(df.head())
    print("\n--- MISSING VALUES ---")
    print(df.isnull().sum())
    print("\n--- DUPLICATES ---")
    print(df.duplicated().sum())


In [34]:
def clean_column_names(df):
    df.columns = df.columns.str.strip().str.lower()
    return df

In [35]:
def remove_duplicates(df):
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    print(f"\nDuplicates removed: {before - after}")
    return df

In [36]:
def convert_date_features(df):
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["year"] = df["date"].dt.year
        df["month"] = df["date"].dt.month
        df["day"] = df["date"].dt.day
        df["day_of_week"] = df["date"].dt.dayofweek
        df["week_of_year"] = df["date"].dt.isocalendar().week.astype("float")
    return df


In [37]:
def feature_engineering(df):
    if "views" in df.columns:
        df["views"] = df["views"].replace(0, np.nan)

    if {"likes", "comments", "views"}.issubset(df.columns):
        df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"]

    if {"watch_time_minutes", "views"}.issubset(df.columns):
        df["avg_watch_time_per_view"] = df["watch_time_minutes"] / df["views"]

    if {"watch_time_minutes", "video_length_minutes"}.issubset(df.columns):
        df["watch_time_to_length_ratio"] = df["watch_time_minutes"] / (df["video_length_minutes"] + 1)

    if {"views", "subscribers"}.issubset(df.columns):
        df["views_subscriber_ratio"] = df["views"] / (df["subscribers"] + 1)

    if {"likes", "views"}.issubset(df.columns):
        df["like_rate"] = df["likes"] / df["views"]

    if {"comments", "views"}.issubset(df.columns):
        df["comment_rate"] = df["comments"] / df["views"]

    return df

In [38]:
def cap_outliers_iqr(df, numeric_cols):
    df = df.copy()

    for col in numeric_cols:
        if col in df.columns:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

    return df

In [39]:
def save_cleaned_data(df):
    output_path = os.path.join(OUTPUT_DIR, "cleaned_data.csv")
    df.to_csv(output_path, index=False)
    print(f"\nCleaned data saved to: {output_path}")

In [40]:
def run_eda(df):
    print("\n--- DESCRIPTIVE STATISTICS ---")
    print(df.describe(include="all"))

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    if len(numeric_cols) > 0:
        corr = df[numeric_cols].corr()

        plt.figure(figsize=(12, 8))
        plt.imshow(corr, cmap="coolwarm", aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.columns)), corr.columns)
        plt.title("Correlation Matrix")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "correlation_matrix.png"))
        plt.close()

        df[numeric_cols].hist(figsize=(16, 12), bins=30)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "numeric_distributions.png"))
        plt.close()

    if "category" in df.columns:
        plt.figure(figsize=(10, 5))
        df["category"].value_counts().head(10).plot(kind="bar")
        plt.title("Top Video Categories")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "top_categories.png"))
        plt.close()

    if TARGET_COL in df.columns:
        plt.figure(figsize=(8, 5))
        df[TARGET_COL].plot(kind="hist", bins=30)
        plt.title("Target Distribution - ad_revenue_usd")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "target_distribution.png"))
        plt.close()



In [41]:
def prepare_features_target(df, target_col):
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset.")

    drop_cols = []
    if "video_id" in df.columns:
        drop_cols.append("video_id")
    if "date" in df.columns:
        drop_cols.append("date")

    X = df.drop(columns=[target_col] + drop_cols)
    y = df[target_col]

    return X, y

In [42]:
def build_preprocessor(X):
    numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    return preprocessor


In [43]:

def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    return {
        "model": name,
        "r2_score": round(r2, 4),
        "rmse": round(rmse, 4),
        "mae": round(mae, 4),
        "pipeline": pipeline
    }


In [44]:
def get_models():
    return {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=0.001),
        "Random Forest Regressor": RandomForestRegressor(
            n_estimators=150,
            max_depth=12,
            random_state=42,
            n_jobs=-1
        ),
        "Gradient Boosting Regressor": GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )
    }

In [45]:
def save_metrics(results_df):
    metrics_path = os.path.join(OUTPUT_DIR, "model_metrics.csv")
    results_df.to_csv(metrics_path, index=False)
    print(f"\nModel metrics saved to: {metrics_path}")


def save_best_model(best_pipeline):
    joblib.dump(best_pipeline, os.path.join(OUTPUT_DIR, "best_model.pkl"))
    print("\nBest model saved successfully.")

In [46]:
def generate_insights(df, results_df):
    insights = []

    insights.append("PROJECT INSIGHTS")
    insights.append("=" * 50)

    if "category" in df.columns and TARGET_COL in df.columns:
        category_revenue = df.groupby("category")[TARGET_COL].mean().sort_values(ascending=False).head(5)
        insights.append("\nTop 5 categories by average ad revenue:")
        for category, value in category_revenue.items():
            insights.append(f"- {category}: {value:.2f}")

    if "country" in df.columns and TARGET_COL in df.columns:
        country_revenue = df.groupby("country")[TARGET_COL].mean().sort_values(ascending=False).head(5)
        insights.append("\nTop 5 countries by average ad revenue:")
        for country, value in country_revenue.items():
            insights.append(f"- {country}: {value:.2f}")

    if "device" in df.columns and TARGET_COL in df.columns:
        device_revenue = df.groupby("device")[TARGET_COL].mean().sort_values(ascending=False)
        insights.append("\nAverage ad revenue by device:")
        for device, value in device_revenue.items():
            insights.append(f"- {device}: {value:.2f}")

    best_model_name = results_df.sort_values(by="r2_score", ascending=False).iloc[0]["model"]
    insights.append(f"\nBest performing model based on R2: {best_model_name}")

    insight_text = "\n".join(insights)

    with open(os.path.join(OUTPUT_DIR, "model_insights.txt"), "w", encoding="utf-8") as f:
        f.write(insight_text)

    print("\nInsights generated and saved.")
    print("\n" + insight_text)

In [47]:
def main():
    create_output_dir()

    df = load_data(DATA_PATH)
    df = clean_column_names(df)
    basic_inspection(df)

    df = remove_duplicates(df)
    df = convert_date_features(df)
    df = feature_engineering(df)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if TARGET_COL in numeric_cols:
        numeric_cols.remove(TARGET_COL)

    df = cap_outliers_iqr(df, numeric_cols)
    save_cleaned_data(df)
    run_eda(df)

    X, y = prepare_features_target(df, TARGET_COL)
    preprocessor = build_preprocessor(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = get_models()
    results = []

    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        result = evaluate_model(name, pipeline, X_train, X_test, y_train, y_test)
        results.append(result)

        print(f"\n{name}")
        print(f"R² Score: {result['r2_score']}")
        print(f"RMSE: {result['rmse']}")
        print(f"MAE: {result['mae']}")

    results_df = pd.DataFrame([
        {
            "model": r["model"],
            "r2_score": r["r2_score"],
            "rmse": r["rmse"],
            "mae": r["mae"]
        }
        for r in results
    ]).sort_values(by="r2_score", ascending=False)

    print("\n--- MODEL COMPARISON ---")
    print(results_df)

    save_metrics(results_df)

    best_result = sorted(results, key=lambda x: x["r2_score"], reverse=True)[0]
    save_best_model(best_result["pipeline"])

    generate_insights(df, results_df)

    print("\nProject completed successfully.")

if __name__ == "__main__":
    main()

Dataset loaded successfully.
Shape: (122400, 12)

--- DATASET INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122400 entries, 0 to 122399
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   video_id              122400 non-null  object 
 1   date                  122400 non-null  object 
 2   views                 122400 non-null  int64  
 3   likes                 116283 non-null  float64
 4   comments              116288 non-null  float64
 5   watch_time_minutes    116295 non-null  float64
 6   video_length_minutes  122400 non-null  float64
 7   subscribers           122400 non-null  int64  
 8   category              122400 non-null  object 
 9   device                122400 non-null  object 
 10  country               122400 non-null  object 
 11  ad_revenue_usd        122400 non-null  float64
dtypes: float64(5), int64(2), object(5)
memory usage: 11.2+ MB
None

--- FIRST 5 ROWS ---
